### Tensor基本操作
Tensor是PyTorch的核心数据结构，可以把它理解成“能在GPU上跑、自带求导功能的NumPy数组”

In [ ]:
import torch
import numpy as np

# ===== 创建Tensor =====
a = torch.tensor([1.0, 2.0, 3.0])  # 从列表创建
b = torch.zeros(3, 4)              # 全零矩阵
c = torch.ones(2, 3)               # 全一
d = torch.rand(3, 3)               # 0到1均匀分布随机
e = torch.randn(3, 3)              # 标准正态分布
f = torch.arange(0, 10, 2)         # 类似于np.arange

display("形状：", a.shape)
display("数据类型：", a.dtype)
display("所在设备：", a.device)

# ===== Tensor与NumPy互转 =====
np_arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(np_arr)  # numpy转tensor
back = t.numpy()              #  tensor转numpy
display("NumPy转Tensor：", t)
display("Tensor转NumPy：", back)

In [ ]:
# 基本运算
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

display("逐元素加法：", x + y)
display("逐元素乘法：", x * y)
display("矩阵乘法：", x @ y)
display("转置：", x.T)

# reshape与view
z = torch.arange(12).float()
display("原始", z.shape)
display("reshape成3×4：", z.reshape(3, 4))
display("展平：", z.reshape(-1))  # -1 是 PyTorch/NumPy 中的便捷写法，表示"自动推断长度"，无需手动计算总元素个数。

display("求和：", x.sum())
display("按行求和：", x.sum(dim=1))
display("按列求和：", x.sum(dim=0))
display("最大值：", x.max())
display("最大值位置：", x.argmax())
display("均值：", x.mean())

## autograd自动微分
这是PyTorch最重要的特性，整个神经网络的训练都依赖它，原理是：PyTorch会记录你对Tensor做的所有操作，然后用链式法则自动计算梯度。

In [ ]:
# 只有requires_grad=Ture的Tensor才会被追踪梯度
x = torch.tensor(3.0, requires_grad=True)

# 定义一个计算：y = x^2 + 2x + 1
y = x**2 + 2*x + 1
display("y = ", y)

# 反向传播：自动计算dy/dx
y.backward()

# 查看梯度：dy/dx = 2x + 2
display("dy/dx = ", x.grad)

In [ ]:
# 多变量的梯度
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

# z = a^2b + b^3,对a求导是2ab，对吧求导是a^2+3b^2
z = a**2 * b + b**3
z.backward()

display('dz/da = ', a.grad)
display('dz/db = ', b.grad)

In [ ]:
# 梯度会累积，每次反向传播前要清零
x = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y= x * 2
    y.backward()
    display(f"第{i+1}次梯度：{x.grad}")
    # 若不清零，梯度会一直叠加

In [ ]:
# 正确做法：每次backward之前先清零
x = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y = x * 2
    y.backward()
    display(f"第{i+1}次梯度: {x.grad}")   # 每次都是2.0
    x.grad.zero_()                        # 清零！下划线表示原地操作

In [ ]:
# 手写线性回归
import torch
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ===== 第一步：生成模拟数据 =====
# 真实关系：y = 2x + 1 + 噪声
torch.manual_seed(42)                          # 固定随机种子，结果可复现
X = torch.randn(100, 1)                        # 100个样本，1个特征
y = 2 * X + 1 + 0.3 * torch.randn(100, 1)    # 真实权重w=2，偏置b=1

display("X形状:", X.shape)
display("y形状:", y.shape)

# 可视化数据
plt.scatter(X.numpy(), y.numpy(), alpha=0.5)
plt.xlabel("X")
plt.ylabel("y")
plt.title("模拟数据")
plt.show()

# ===== 第二步：初始化参数 =====
# 我们要让模型自己学出 w≈2, b≈1
w = torch.tensor(0.0, requires_grad=True)   # 初始权重为0
b = torch.tensor(0.0, requires_grad=True)   # 初始偏置为0

display(f"初始参数: w={w.item():.4f}, b={b.item():.4f}")

# ===== 第三步：训练循环 =====
learning_rate = 0.1
epochs = 100
loss_history = []

for epoch in range(epochs):
    # 1. 前向传播：计算预测值
    y_pred = w * X + b

    # 2. 计算损失（均方误差MSE）
    loss = ((y_pred - y) ** 2).mean()
    loss_history.append(loss.item())

    # 3. 反向传播：计算梯度
    loss.backward()

    # 4. 更新参数（梯度下降）
    # 注意：参数更新不需要被追踪梯度，所以用torch.no_grad()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # 5. 清零梯度（重要！）
    w.grad.zero_()
    b.grad.zero_()

    # 每10轮打印一次
    if (epoch + 1) % 10 == 0:
        display(f"Epoch {epoch+1:3d} | Loss: {loss.item():.6f} | w: {w.item():.4f} | b: {b.item():.4f}")

display(f"训练完成！最终参数: w={w.item():.4f}, b={b.item():.4f}")
display(f"真实参数: w=2.0000, b=1.0000")

# ===== 第四步：可视化结果 =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 左图：拟合效果
ax1.scatter(X.numpy(), y.numpy(), alpha=0.5, label="真实数据")
X_line = torch.linspace(-3, 3, 100).reshape(-1, 1)
with torch.no_grad():
    y_line = w * X_line + b
ax1.plot(X_line.numpy(), y_line.numpy(), 'r-', linewidth=2, label=f"拟合直线: y={w.item():.2f}x+{b.item():.2f}")
ax1.legend()
ax1.set_title("线性回归拟合效果")

# 右图：损失曲线
ax2.plot(loss_history)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("训练损失曲线")

plt.tight_layout()
plt.show()

用nn.Module搭建神经网络

In [ ]:
import torch
import torch.nn as nn

# nn.Module，所有神经网络层都继承自nn.Module，最简单的例子：单层线性变换

linear = nn.Linear(in_features=3, out_features=2)  # 输入三维，输出2维
display("权重形状：", linear.weight.shape)
display("偏置形状：", linear.bias.shape)

x = torch.randn(5, 3)  # 5个样本，每个3维
out = linear(x)
display("输入形状：", x.shape)
display("输出形状：", out.shape)

In [ ]:
# 搭建一个MLP(多层感知机)
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):  # 输入维度、隐藏层维度、输出维度
        super().__init__()                                  # 调用父类nn.Module的构造函数，Pytorch的固定写法
        #定义网络层
        self.layer1 = nn.Linear(input_dim, hidden_dim)      # 第一层，全连接层，input_dim -> hidden_dim
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()                               # 激活函数ReLU,公式为f(x) = max(0, x)，引入非线性
        self.dropout = nn.Dropout(p=0.1)                    # Dropout正则化：训练时随机将10%的神经元输出置为0，防止过拟合

    def forward(self, x):
        # 定义前向传播过程
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.layer3(X)
        return x
    
# 实例化模型
model = MLP(input_dim=784, hidden_dim=256, output_dim=10)
display(model)      # 打印网络结构
display("参数总量：")
total = sum(p.numel() for p in model.parameters())
display(f"{total:,} 个参数")


In [ ]:
# 常用激活函数对比
x = torch.linspace(-3, 3, 100)

relu = nn.ReLU()(x)
gelu = nn.GELU()(x)
sigmoid = nn.Sigmoid()(x)
tanh = nn.Tanh()(x)

plt.figure(figsize=(10, 4))
plt.plot(x.numpy(), relu.numpy(), label='ReLU')
plt.plot(x.numpy(), gelu.numpy(), label='GELU')
plt.plot(x.numpy(), sigmoid.detach().numpy(), label='Sigmoid')
plt.plot(x.numpy(), tanh.detach().numpy(), label='Tanh')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.legend()
plt.title("常用激活函数对比")
plt.grid(True, alpha=0.3)
plt.show()

训练MNIST手写数字分类

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 第一步：加载数据
# transforms：对图片做预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 下载并加载训练集和测试集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

display(f"训练集大小：{len(train_dataset)}")
display(f"测试集大小：{len(test_dataset)}")

# 数据模样
images, labels = next(iter(train_loader))
display(f"一个batch的图片形状: {images.shape}")
display(f"一个batch的标签形状: {labels.shape}")

In [ ]:
# 可视化几张图片
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f"标签：{labels[i].item()}")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 第二步：定义模型
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)
    
model = MNISTClassifier()
display(model)

In [ ]:
# 第三步：定义损失函数和优化器
criterion = nn.CrossEntropyLoss()   # 多分类任务用交叉熵损失
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 测试一下前向传播是否正常
sample_input = torch.randn(64, 1, 28, 28)
sample_output = model(sample_input)
display("输出形状：", sample_output.shape)

In [ ]:
# 第四步：训练循环
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        # 1.前向传播
        outputs = model(images)
        loss = criterion(outputs, labels)

        # 2.反向传播
        optimizer.zero_grad()   # 清零梯度
        loss.backward()         # 计算梯度
        optimizer.step()        # 更新参数

        # 统计准确率
        total_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [ ]:
# 第五步：开始训练
epochs = 5
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    display(f"Epoch {epoch+1}/{epochs} | "
            f"训练损失：{train_loss:.4f} | 训练准确率：{train_acc:.4f} | "
            f"测试损失：{test_loss:.4f} | 测试准确率：{test_acc:.4f}")
    
# 第六步：可视化训练过程
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='训练损失')
ax1.plot(test_losses,  label='测试损失')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('损失曲线')
ax1.legend()

ax2.plot(train_accs, label='训练准确率')
ax2.plot(test_accs,  label='测试准确率')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('准确率曲线')
ax2.legend()

plt.tight_layout()
plt.show()

display(f"最终测试准确率: {test_accs[1]:.4f}")
display(f"正确分类了 {int(test_accs[1]*10000)} / 10000 张图片")